# Turning Words into Numbers
Computers understand numbers, not words. Let's see the different ways to convert text into numbers.

## Setup

In [1]:
!pip install -q gensim

In [2]:
import pandas as pd
import numpy as np

## 1. Our Sentences

In [3]:
sentences = [
    "the cat drinks milk",
    "the dog drinks water",
    "the cat chases the mouse",
    "the dog chases the cat"
]

for sentence in sentences:
    print(sentence)

the cat drinks milk
the dog drinks water
the cat chases the mouse
the dog chases the cat


## 2. One Hot Encoding

In [4]:
vocabulary = sorted(set(" ".join(sentences).split()))
print(vocabulary)
print("Vocabulary size:", len(vocabulary))

['cat', 'chases', 'dog', 'drinks', 'milk', 'mouse', 'the', 'water']
Vocabulary size: 8


In [5]:
one_hot = pd.get_dummies(vocabulary).astype(int)
one_hot.index = vocabulary
one_hot

,cat,chases,dog,drinks,milk,mouse,the,water
cat,1,0,0,0,0,0,0,0
chases,0,1,0,0,0,0,0,0
dog,0,0,1,0,0,0,0,0
drinks,0,0,0,1,0,0,0,0
milk,0,0,0,0,1,0,0,0
mouse,0,0,0,0,0,1,0,0
the,0,0,0,0,0,0,1,0
water,0,0,0,0,0,0,0,1


In [6]:
for word in "the cat drinks milk".split():
    print(f"{word:<8}", one_hot.loc[word].values)

the      [0 0 0 0 0 0 1 0]
cat      [1 0 0 0 0 0 0 0]
drinks   [0 0 0 1 0 0 0 0]
milk     [0 0 0 0 1 0 0 0]


## 3. Bag of Words

In [7]:
from sklearn.feature_extraction.text import CountVectorizer

bow = CountVectorizer()
bow_matrix = bow.fit_transform(sentences)

pd.DataFrame(bow_matrix.toarray(), columns=bow.get_feature_names_out(), index=sentences)

,cat,chases,dog,drinks,milk,mouse,the,water
the cat drinks milk,1,0,0,1,1,0,1,0
the dog drinks water,0,0,1,1,0,0,1,1
the cat chases the mouse,1,1,0,0,0,1,2,0
the dog chases the cat,1,1,1,0,0,0,2,0


In [8]:
new_sentence = ["the cat drinks water and the dog drinks milk"]
pd.DataFrame(bow.transform(new_sentence).toarray(), columns=bow.get_feature_names_out(), index=new_sentence)

,cat,chases,dog,drinks,milk,mouse,the,water
the cat drinks water and the dog drinks milk,1,0,1,2,1,0,2,1


## 4. TF-IDF

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(sentences)

pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf.get_feature_names_out(), index=sentences).round(2)

,cat,chases,dog,drinks,milk,mouse,the,water
the cat drinks milk,0.42,0.00,0.00,0.52,0.66,0.00,0.34,0.00
the dog drinks water,0.00,0.00,0.50,0.50,0.00,0.00,0.33,0.63
the cat chases the mouse,0.36,0.45,0.00,0.00,0.00,0.57,0.59,0.00
the dog chases the cat,0.39,0.48,0.48,0.00,0.00,0.00,0.63,0.00


In [10]:
pd.DataFrame({"word": tfidf.get_feature_names_out(), "idf": tfidf.idf_.round(2)}).sort_values("idf")

,word,idf
6,the,1.00
0,cat,1.22
2,dog,1.51
1,chases,1.51
3,drinks,1.51
4,milk,1.92
5,mouse,1.92
7,water,1.92


## 5. Word2Vec

In [11]:
corpus = [
    "the king and the queen live in the palace",
    "the queen and the king wear a golden crown",
    "the prince and the princess play in the palace",
    "the king the queen and the prince rule the kingdom",
    "the princess and the queen wear a crown",
    "the cat and the dog are friendly pets",
    "the dog and the cat play with the ball",
    "the cat drinks milk and the dog drinks milk",
    "the puppy and the kitten are cute pets",
    "the dog the cat and the puppy sleep at home",
    "apple and mango are sweet fruits",
    "banana and orange are tasty fruits",
    "mango and apple and banana are healthy fruits",
    "we eat orange and mango and apple every day",
    "children love sweet fruits like banana and mango"
]

tokens = [sentence.split() for sentence in corpus]
print(tokens[:3])

[['the', 'king', 'and', 'the', 'queen', 'live', 'in', 'the', 'palace'], ['the', 'queen', 'and', 'the', 'king', 'wear', 'a', 'golden', 'crown'], ['the', 'prince', 'and', 'the', 'princess', 'play', 'in', 'the', 'palace']]


### 5a. CBOW (Continuous Bag of Words)
Uses the surrounding words to guess the middle word.

In [12]:
from gensim.models import Word2Vec

cbow = Word2Vec(tokens, vector_size=10, window=3, min_count=1, sg=0, epochs=1000, seed=42, workers=1)

print(cbow.wv["king"])

[ 0.6665086   1.2381063   0.8404134  -1.065451    1.2672179  -0.33869624
 -0.48958814 -0.3258633  -0.5590634  -0.3753418 ]


In [13]:
print(cbow.wv.most_similar("king", topn=3))
print(cbow.wv.most_similar("cat", topn=3))
print(cbow.wv.most_similar("apple", topn=3))

[('queen', 0.9918655157089233), ('rule', 0.9769662022590637), ('prince', 0.9722539186477661)]
[('dog', 0.9852689504623413), ('ball', 0.9587697386741638), ('with', 0.9410818219184875)]
[('orange', 0.9880264401435852), ('mango', 0.9861533045768738), ('eat', 0.9711362719535828)]


### 5b. Skip-gram
Uses the middle word to guess the surrounding words.

In [14]:
skipgram = Word2Vec(tokens, vector_size=10, window=3, min_count=1, sg=1, epochs=1000, seed=42, workers=1)

print(skipgram.wv["king"])

[ 0.5239351   0.9830707   0.9886151  -0.86254793  1.4568605  -0.16490375
 -0.66756046 -0.24948122 -0.50937104 -0.347343  ]


In [15]:
print(skipgram.wv.most_similar("king", topn=3))
print(skipgram.wv.most_similar("cat", topn=3))
print(skipgram.wv.most_similar("apple", topn=3))

[('queen', 0.9865411520004272), ('wear', 0.9658909440040588), ('crown', 0.9602246284484863)]
[('dog', 0.9892785549163818), ('ball', 0.9056107401847839), ('sleep', 0.9017060995101929)]
[('orange', 0.9792715907096863), ('mango', 0.9716011881828308), ('day', 0.9371757507324219)]


### 5c. CBOW vs Skip-gram

In [16]:
pairs = [("king", "queen"), ("cat", "dog"), ("apple", "mango"), ("king", "banana"), ("cat", "mango")]

print(f"{'Pair':<18}{'CBOW':<10}{'Skip-gram':<10}")
print("-" * 38)

for a, b in pairs:
    print(f"{a + ' - ' + b:<18}{cbow.wv.similarity(a, b):<10.2f}{skipgram.wv.similarity(a, b):<10.2f}")

Pair              CBOW      Skip-gram 
--------------------------------------
king - queen      0.99      0.99      
cat - dog         0.99      0.99      
apple - mango     0.99      0.97      
king - banana     0.12      0.15      
cat - mango       0.17      0.23      


## 6. GloVe (Pre-trained Word Vectors)

In [17]:
import gensim.downloader as api

glove = api.load("glove-wiki-gigaword-50")

[==================================================] 100.0% 66.0/66.0MB downloaded


In [18]:
print(glove["computer"])
print("Vector size:", len(glove["computer"]))

[ 0.079084 -0.81504   1.7901    0.91653   0.10797  -0.55628  -0.84427
 -1.4951    0.13418   0.63627   0.35146   0.25813  -0.55029   0.51056
  0.37409   0.12092  -1.6166    0.83653   0.14202  -0.52348   0.73453
  0.12207  -0.49079   0.32533   0.45306  -1.585    -0.63848  -1.0053
  0.10454  -0.42984   3.181    -0.62187   0.16819  -1.0139    0.064058
  0.57844  -0.4556    0.73783   0.37203  -0.57722   0.66441   0.055129
  0.037891  1.3275    0.30991   0.50697   1.2357    0.1274   -0.11434
  0.20709 ]
Vector size: 50


In [19]:
print(glove.most_similar("cricket", topn=5))

[('indies', 0.8089948296546936), ('twenty20', 0.8088222146034241), ('rugby', 0.8044811487197876), ('cricketers', 0.7978313565254211), ('england', 0.7800653576850891)]


In [20]:
print(glove.most_similar("mango", topn=5))

[('pineapple', 0.9108651280403137), ('guava', 0.8952961564064026), ('papaya', 0.8908727169036865), ('pear', 0.854493260383606), ('coconut', 0.8482269048690796)]


In [21]:
print("cat - dog   :", round(glove.similarity("cat", "dog"), 2))
print("cat - car   :", round(glove.similarity("cat", "car"), 2))
print("happy - glad:", round(glove.similarity("happy", "glad"), 2))

cat - dog   : 0.92
cat - car   : 0.36
happy - glad: 0.87


### Word Maths: king - man + woman = ?

In [22]:
print(glove.most_similar(positive=["king", "woman"], negative=["man"], topn=1))

[('queen', 0.8523604273796082)]


In [23]:
print(glove.most_similar(positive=["paris", "india"], negative=["france"], topn=1))

[('delhi', 0.8888585567474365)]


In [24]:
print(glove.doesnt_match(["apple", "mango", "banana", "car"]))

car
